# DistilBERT — tool-call classifier

Trains A (tool call only) and B (call + user request + agent history) on the full training split (335,349 rows), scores them on val, test and the two outside test sets, and publishes them to the Hugging Face Hub.

Run on Colab with an A100 GPU: `Runtime > Run all`. Code comes from this repo, data from [LucidAI-Dataset](https://github.com/Enes-Ozbek/LucidAI-Dataset); outputs go to Google Drive (`MyDrive/mayi_distilbert/`). If Colab disconnects, rerun the same cell: training resumes from the last checkpoint.

In [3]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
NVIDIA A100-SXM4-40GB, 40960 MiB


In [4]:
from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/mayi_distilbert'
!mkdir -p $OUT

Mounted at /content/drive


In [5]:
# Code: this repo's scripts. Data: the cleaned dataset.
%cd /content
!test -d mayi-distilbert || git clone -q https://github.com/akifkulll/mayi-distilbert
!test -d LucidAI-Dataset || git clone -q https://github.com/Enes-Ozbek/LucidAI-Dataset
%cd /content/mayi-distilbert
DATA = '/content/LucidAI-Dataset/data'
!ls $DATA

/content
/content/mayi-distilbert
dropped.jsonl.gz	    REPORT.md	 train_1.csv.gz  val.csv.gz
ood_approve_or_deny.csv.gz  stats.json	 train_2.csv.gz
ood_saroku.csv.gz	    test.csv.gz  train_3.csv.gz


## B: call + request + history, 335k rows

In [7]:
!python train_distilbert.py --data-dir $DATA --input b --train-rows 0 --batch-size 32 --eval-every 1000 --out $OUT/distilbert_b_full

device=cuda  input=input_b  model=distilbert-base-uncased
train rows=335349  val rows (during training)=5001  deny share=0.506  load 16s
config.json: 100% 483/483 [00:00<00:00, 2.15MB/s]
tokenizer_config.json: 100% 48.0/48.0 [00:00<00:00, 223kB/s]
vocab.txt: 100% 232k/232k [00:00<00:00, 761kB/s]
tokenizer.json: 100% 466k/466k [00:00<00:00, 3.74MB/s]

model.safetensors: downloading bytes:  64% 173M/268M [00:01<00:00, 176MB/s, 10.5MB/s  ]
model.safetensors: downloading bytes:  82% 220M/268M [00:01<00:00, 226MB/s, 16.4MB/s  ]
model.safetensors: downloading bytes: 100% 250M/250M [00:02<00:00, 113MB/s, 23.2MB/s  ]
model.safetensors: reconstructing file: 100% 268M/268M [00:02<00:00, 121MB/s, 25.3MB/s  ]
Loading weights: 100% 100/100 [00:00<00:00, 5589.28it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias   

## A: call only, 335k rows

In [8]:
!python train_distilbert.py --data-dir $DATA --input a --train-rows 0 --eval-every 1000 --out $OUT/distilbert_a_full

device=cuda  input=input_a  model=distilbert-base-uncased
train rows=335349  val rows (during training)=5001  deny share=0.506  load 16s
Loading weights: 100% 100/100 [00:00<00:00, 5349.81it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
{'loss': '0.6932', 'grad_

## Score

In [9]:
for run in ['distilbert_a_full', 'distilbert_b_full']:
    for split in ['val', 'test', 'ood_approve_or_deny', 'ood_saroku']:
        !python evaluate_predictions.py --data-dir $DATA --pred $OUT/$run/predictions/{split}.csv --split {split} --out $OUT/results/{run}_{split}

val: acc=0.9141 macro_f1=0.9141 false_allow=0.0908 auroc=0.9692 -> /content/drive/MyDrive/mayi_distilbert/results/distilbert_a_full_val
test: acc=0.9142 macro_f1=0.9142 false_allow=0.0879 auroc=0.9685 -> /content/drive/MyDrive/mayi_distilbert/results/distilbert_a_full_test
ood_approve_or_deny: acc=0.8764 macro_f1=0.8763 false_allow=0.0946 auroc=0.9518 -> /content/drive/MyDrive/mayi_distilbert/results/distilbert_a_full_ood_approve_or_deny
ood_saroku: acc=0.6648 macro_f1=0.6574 false_allow=0.2611 auroc=0.7204 -> /content/drive/MyDrive/mayi_distilbert/results/distilbert_a_full_ood_saroku
val: acc=0.9614 macro_f1=0.9614 false_allow=0.0415 auroc=0.9927 -> /content/drive/MyDrive/mayi_distilbert/results/distilbert_b_full_val
test: acc=0.9607 macro_f1=0.9607 false_allow=0.0405 auroc=0.9929 -> /content/drive/MyDrive/mayi_distilbert/results/distilbert_b_full_test
ood_approve_or_deny: acc=0.9220 macro_f1=0.9216 false_allow=0.0961 auroc=0.9760 -> /content/drive/MyDrive/mayi_distilbert/results/dist

## Comparison (test split)

In [ ]:
import json, os, pandas as pd
rows = []
for run, n in [('distilbert_a_full', '335k'), ('distilbert_b_full', '335k')]:
    p = f'{OUT}/results/{run}_test/metrics.json'
    if not os.path.exists(p):
        continue
    r = json.load(open(p)); o = r['overall']; both = r['groups']['call_has_both_labels']['True']
    row = {'model': run, 'train rows': n, 'accuracy': o['accuracy'], 'macro_f1': o['macro_f1'],
           'false_allow': o['false_allow_rate'], 'false_deny': o['false_deny_rate'], 'auroc': o['auroc'],
           'both_labels_f1': both['macro_f1']}
    for split in ['ood_approve_or_deny', 'ood_saroku']:
        q = f'{OUT}/results/{run}_{split}/metrics.json'
        if os.path.exists(q):
            row[split + '_f1'] = json.load(open(q))['overall']['macro_f1']
    rows.append(row)
pd.DataFrame(rows).round(4)

## Download the new results
Without `errors.csv` (GitHub's secret scanning flags a fake key in the synthetic data).

In [11]:
import shutil, glob
dst = '/content/results_full'
shutil.rmtree(dst, ignore_errors=True); os.makedirs(dst)
for run in ['distilbert_a_full', 'distilbert_b_full']:
    if os.path.exists(f'{OUT}/{run}/run.json'):
        shutil.copy(f'{OUT}/{run}/run.json', f'{dst}/{run}_run.json')
for folder in glob.glob(f'{OUT}/results/distilbert_*_full_*'):
    name = os.path.basename(folder); os.makedirs(f'{dst}/{name}')
    for f in ['metrics.md', 'metrics.json', 'confusion_matrix.png']:
        if os.path.exists(f'{folder}/{f}'):
            shutil.copy(f'{folder}/{f}', f'{dst}/{name}/{f}')
shutil.make_archive('/content/results_full', 'zip', dst)
from google.colab import files
files.download('/content/results_full.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Publish to the Hugging Face Hub

In [12]:
from huggingface_hub import login
login()

In [13]:
!python push_to_hub.py --model-dir $OUT/distilbert_b_full/model --repo akifkulll/mayi-distilbert-b --input b --results $OUT/results/distilbert_b_full_test/metrics.json
!python push_to_hub.py --model-dir $OUT/distilbert_a_full/model --repo akifkulll/mayi-distilbert-a --input a --results $OUT/results/distilbert_a_full_test/metrics.json

Found 6 files to upload
  Preparing   ████████████████████  6 / 6 ✓
  Uploading   ████████████████████  -
  Committing  ░░░░░░░░░░░░░░░░░░░░  0 / 6
  Preparing   ████████████████████  6 / 6 ✓
  Uploading   ░░░░░░░░░░░░░░░░░░░░  0 / 2 files
  Committing  ░░░░░░░░░░░░░░░░░░░░  0 / 6
  Preparing   ████████████████████  6 / 6 ✓
  Uploading   ░░░░░░░░░░░░░░░░░░░░  0 / 2 files
  Committing  ░░░░░░░░░░░░░░░░░░░░  0 / 6
  Preparing   ████████████████████  6 / 6 ✓
  Uploading   ░░░░░░░░░░░░░░░░░░░░  0 / 2 files
  Committing  ░░░░░░░░░░░░░░░░░░░░  0 / 6
  Preparing   ████████████████████  6 / 6 ✓
  Uploading   ░░░░░░░░░░░░░░░░░░░░  0 / 2 files
  Committing  ░░░░░░░░░░░░░░░░░░░░  0 / 6
  Preparing   ████████████████████  6 / 6 ✓
  Uploading   ░░░░░░░░░░░░░░░░░░░░  0 / 2 files  575kB · 1.15MB/s
  Committing  ░░░░░░░░░░░░░░░░░░░░  0 / 6
  Preparing   ████████████████████  6 / 6 ✓
  Uploading   ░░░░░░░░░░░░░░░░░░░░  0 / 2 files  1.15MB · 1.15MB/s
  Committing  ░░░░░░░░░░░░░░░░░░░░  0 / 6
  Preparing